# Monthly Walk-Forward Rebalance Patch

This notebook is a safe patch notebook for the Black-Litterman research flow.

What it changes:
- adds the missing extra assets back into the investable universe:
  - TEFAS gold
  - TEFAS silver
  - TEFAS platinum
  - BIST 100
- replaces a single static portfolio with a monthly walk-forward rebalance
- reports:
  - chosen assets each month
  - weights each month
  - CAGR
  - volatility
  - Sharpe
  - max drawdown

Important:
- I could not verify the old v10 TEFAS ticker codes automatically, so the 3 TEFAS symbols below are left as placeholders on purpose.
- Replace those placeholders with the exact symbols you used in v10 before running the notebook end-to-end.
- This notebook assumes you already have a daily `returns` DataFrame and an `asset_map` dict, or that you will adapt the loading cell below to your project.


In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pypfopt import expected_returns, risk_models, EfficientFrontier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1) Restore the extra assets

Replace the 3 TEFAS placeholders with the exact codes from the older notebook version.
If your OpenBB or local data loader uses a different BIST100 symbol, change that too.


In [ ]:
# Replace these placeholders with the exact codes from v10
extra_assets = {
    "TEFAS_GOLD": "REPLACE_WITH_V10_GOLD_CODE",
    "TEFAS_SILVER": "REPLACE_WITH_V10_SILVER_CODE",
    "TEFAS_PLATINUM": "REPLACE_WITH_V10_PLATINUM_CODE",
    "BIST100": "XU100.IS",
}

# This notebook expects asset_map to already exist in your research flow.
# If it does not, define it before this cell.
if "asset_map" not in globals():
    asset_map = {}

asset_map.update(extra_assets)
asset_map


## 2) Make sure `returns` exists

This patch notebook expects a daily returns DataFrame called `returns`:
- index: daily datetime index
- columns: asset names that match the keys used in `asset_map`

If your original notebook already creates `returns`, keep using it.
The fallback cell below only helps validate the structure.


In [ ]:
# Optional structure check
if "returns" not in globals():
    raise ValueError(
        "This patch notebook expects a daily `returns` DataFrame from the original research flow. "
        "Load/build your price data first, then re-run this notebook."
    )

returns = returns.copy()
returns.index = pd.to_datetime(returns.index)
returns = returns.sort_index()

display(returns.head())
print("Shape:", returns.shape)


## 3) Portfolio builder

Drop your existing Black-Litterman view construction into this function if you already have custom views.
Right now this uses historical mean returns plus sample covariance as a clean default wrapper around the optimizer.


In [ ]:
def build_portfolio(train_returns: pd.DataFrame):
    train_returns = train_returns.dropna(axis=1, how="all").dropna(how="any")
    if train_returns.shape[1] < 2:
        raise ValueError("Need at least 2 assets with valid training data to build a portfolio.")

    prices = (1 + train_returns).cumprod()

    mu = expected_returns.mean_historical_return(prices, frequency=252)
    S = risk_models.sample_cov(prices, frequency=252)

    ef = EfficientFrontier(mu, S)
    ef.max_sharpe()
    raw_w = ef.clean_weights()

    weights = {k: float(v) for k, v in raw_w.items() if v is not None and float(v) > 0}
    chosen_assets = list(weights.keys())

    return chosen_assets, weights


## 4) Monthly walk-forward rebalance

This is the realistic version:
- use only information known up to each rebalance date
- rebuild the portfolio at each month-end
- hold it through the next month
- repeat


In [ ]:
lookback_window = 252
lookback_months = 12

rebal_dates = returns.resample("M").last().index

selected_history = []
weight_history = []
portfolio_rets = []

for i in range(lookback_months, len(rebal_dates) - 1):
    rebalance_date = rebal_dates[i]
    next_rebalance_date = rebal_dates[i + 1]

    train = returns.loc[:rebalance_date].tail(lookback_window)

    valid_cols = train.columns[train.notna().sum() > max(60, lookback_window // 3)]
    train = train[valid_cols].dropna(how="any", axis=0)

    if train.shape[1] < 2 or train.empty:
        continue

    try:
        chosen_assets, weights = build_portfolio(train)
    except Exception as e:
        print(f"Skipped {rebalance_date.date()} due to optimizer error: {e}")
        continue

    selected_history.append(pd.Series(chosen_assets, name=rebalance_date))
    weight_history.append(pd.Series(weights, name=rebalance_date))

    next_month_slice = returns.loc[
        (returns.index > rebalance_date) & (returns.index <= next_rebalance_date),
        list(weights.keys())
    ].copy()

    if next_month_slice.empty:
        continue

    next_month_slice = next_month_slice.fillna(0.0)
    aligned_w = pd.Series(weights).reindex(next_month_slice.columns).fillna(0.0)
    realized = next_month_slice.mul(aligned_w, axis=1).sum(axis=1)

    portfolio_rets.append(realized)

if not portfolio_rets:
    raise ValueError("No portfolio return series was produced. Check your data coverage and asset symbols.")

portfolio_rets = pd.concat(portfolio_rets).sort_index()
selected_df = pd.DataFrame(selected_history)
weights_df = pd.DataFrame(weight_history).fillna(0.0)
equity_curve = (1 + portfolio_rets).cumprod()

display(selected_df.tail())
display(weights_df.tail())


## 5) Performance summary


In [ ]:
ann_ret = (1 + portfolio_rets).prod() ** (252 / len(portfolio_rets)) - 1
ann_vol = portfolio_rets.std() * np.sqrt(252)
sharpe = ann_ret / ann_vol if ann_vol != 0 else np.nan
max_dd = (equity_curve / equity_curve.cummax() - 1).min()

summary = pd.Series({
    "CAGR": ann_ret,
    "Volatility": ann_vol,
    "Sharpe": sharpe,
    "Max Drawdown": max_dd,
    "Start": portfolio_rets.index.min(),
    "End": portfolio_rets.index.max(),
    "Rebalance Count": len(weights_df),
})

display(summary.to_frame("value"))


## 6) What it chose each month


In [ ]:
display(selected_df)


## 7) Weights each month


In [ ]:
display(weights_df.round(4))


## 8) Equity curve


In [ ]:
plt.figure(figsize=(12, 5))
equity_curve.plot()
plt.title("Monthly Rebalanced Portfolio Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.grid(True, alpha=0.3)
plt.show()
